# Build distance matrix

Outcomes: 

Matrix of pairwise distances between mobility sequences and ATUS sequences

- Rows should index mobility sequences and columns should index ATUS sequences 
- Ideally, the matrix may contain one column indicating the GEOID of each mobility sequence

In [1]:
import numpy as np
import pandas as pd
from weighted_levenshtein import lev

### 1. Load data

In [ ]:
# load atus sequences
atus_seq = pd.read_csv('data/processed/atus_seq.csv', index_col=0)

# load mobility sequences
### **(here, only as example we will use atus sequences in place of mobility sequences
### due to data privacy polgicies. Replace the file with actual mobility sequences)**
mob_seq = pd.read_csv('data/processed/atus_seq.csv', index_col=0)

### 2. Map states into letter alphabet

We will be using the weighted_levenshtein package to compute distances which is very fast. This package requires sequences to have a letter alphabet so we will first map the sequence states into appropriate alphabetical labels.

In this example, we will also collapse the states further into "home", "work", "other places".

In [ ]:
tewhere_map_labels = {
    1: "home",
    2: "work",
    3: "other place",
    4: "restaurants/bars",
    5: "stores/mall",
    6: "school",
    7: "outdoors",
    8: "transport"
}

map_to_alpha = {
    1: 'A',
    2: 'B',
    3: 'C',
    4: 'C',
    5: 'C',
    6: 'C',
    7: 'C',
    8: 'C'
}


atus_seq_alpha = atus_seq.applymap(lambda x: map_to_alpha[x]).astype(str).agg(''.join, axis=1).values
mob_seq_alpha = mob_seq.applymap(lambda x: map_to_alpha[x]).astype(str).agg(''.join, axis=1).values

# peek the first 5 sequences
atus_seq_alpha[:5]

array(['AAAAAAACBBBBBBBBBCBBBBBBBBAAAAAAAAAAAAAAAAAAAAAA',
       'AAAAAAAAAAAAAAACCCCCCCCCAAACAAAAAAAAAAAAAAAAAAAA',
       'AAAAAAAAAAAAACCCAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA',
       'AAAAAAAAAAAAAACCCCCCCCAAAAAAAAAAAAAAAAAAAAAAAAAA',
       'AAAAAAAAACAAACCCCCCCCCAAAAAAACCAAAAAAAAAAAAAAAAA'], dtype=object)

### 3. Make alignment costs

Make indel and substitution alignment costs that will be compliant to the requirements of weighted_levenshtein. The function below shows an example for 3 different alignment cost schemes: Levenshtein I, Levenshtein II, Hamming.

In [10]:
# make distance matrices compliant to weighted_levenshtein

def make_costs(dist_type):
    if dist_type == 'lev1':
        indel_factor = 1
        sub_factor = 1
    elif dist_type == 'lev2':
        indel_factor = 1
        sub_factor = 999999
    elif dist_type == 'hamming':
        indel_factor = 999999
        sub_factor = 1
    
    indel_costs = np.ones(128, dtype=np.float64) * indel_factor
    substitute_costs = np.ones((128, 128), dtype=np.float64) * sub_factor

    return indel_costs, substitute_costs

In [11]:
# test cases to validate the alignment costs

indel_costs, substitute_costs = make_costs('hamming')
assert lev('AAAABBBB', 'ABBCCDEE', insert_costs=indel_costs, delete_costs=indel_costs, substitute_costs=substitute_costs) == 7

indel_costs, substitute_costs = make_costs('lev1')
assert lev('AAAABBBB', 'ABBCCDEE', insert_costs=indel_costs, delete_costs=indel_costs, substitute_costs=substitute_costs) == 7

indel_costs, substitute_costs = make_costs('lev2')
assert lev('AAAABBBB', 'ABBCCDEE', insert_costs=indel_costs, delete_costs=indel_costs, substitute_costs=substitute_costs) == 10

### 4. Build distance matrix

Note: for a very large mobility sample, the following code can probably be optimized

In [12]:
n_mob = len(mob_seq_alpha)
n_atus = len(atus_seq_alpha)

indel_costs, substitute_costs = make_costs('hamming')
dist_matrix = np.zeros((n_mob,n_atus))
for j, s1 in enumerate(mob_seq_alpha):          # row index
    for i, s2 in enumerate(atus_seq_alpha):     # col index
        dist_matrix[i,j] = lev(s1, s2, insert_costs=indel_costs, delete_costs=indel_costs, substitute_costs=substitute_costs)

In [13]:
dist_matrix

array([[ 0., 19., 19., ..., 19., 20., 13.],
       [19.,  0., 11., ..., 10., 12., 18.],
       [19., 11.,  0., ...,  3., 19., 23.],
       ...,
       [19., 10.,  3., ...,  0., 22., 23.],
       [20., 12., 19., ..., 22.,  0., 15.],
       [13., 18., 23., ..., 23., 15.,  0.]])

In [ ]:
dist_matrix = pd.DataFrame(dist_matrix, index=mob_seq.index, columns=atus_seq.index)
dist_matrix.to_parquet('data/processed/dist_matrix.parquet')